In [1]:
from vector_store_controller.qdrant_db import QdrantDB

c:\Users\samarth.srivastava\Desktop\new_check_proj\test_env2\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [1]:
import sys
sys.path.insert(0, r'C:\Users\samarth.srivastava\Desktop\RAG_comprehensive')

In [264]:
from langchain_core.embeddings.embeddings import Embeddings
from fastembed import SparseTextEmbedding
from langchain_core.documents import Document
from qdrant_client.http.models import SparseVector
from qdrant_client.http import models as rest


class sparse_embed(Embeddings):
    embeddings_sparsed = SparseTextEmbedding(model_name="Qdrant/bm25")
    def embed_documents(self, texts):
        embeds = list(self.embeddings_sparsed.embed(texts))
        return [
            {"indices": e.indices.tolist(), "values": e.values.tolist()}
            for e in embeds
        ]
    def embed_query(self, text):
        e = list(self.embeddings_sparsed.embed(text))[0]
        print({"values":e.values.tolist(), "indices":e.indices.tolist()})
        return rest.NamedSparseVector(
                                    name="text",
                                    vector=rest.SparseVector(
                                    indices=e.indices.tolist(),
                                    values=e.values.tolist(),
                                ),
                            )
        # return [{"values":e.values.tolist(), "indices":e.indices.tolist()}]

Fetching 30 files: 100%|██████████| 30/30 [00:00<?, ?it/s]


In [ ]:
embeddings_sparsed1 = SparseTextEmbedding(model_name="Qdrant/bm25")
from langchain_core.documents import Document
q = Document(page_content='What is this')
e = list(embeddings_sparsed1.embed('alskfdjal akljdfl aldjf '))[0]
[{"indices": e.indices.tolist(), "values": e.values.tolist()}]

Fetching 30 files: 100%|██████████| 30/30 [00:00<?, ?it/s]


[{'indices': [605562940, 699705489, 898824501],
  'values': [1.6786885245901642, 1.6786885245901642, 1.6786885245901642]}]

In [138]:
d

[SparseEmbedding(values=array([1.68774348]), indices=array([683667415]))]

In [ ]:
from langchain_qdrant import Qdrant
from qdrant_client import QdrantClient
from langchain_ollama import OllamaEmbeddings
from qdrant_client.http import models as rest


In [48]:
client = QdrantClient(url='http://localhost:6333')

client.create_collection(collection_name='samarth19', vectors_config={'some_name': rest.VectorParams(size=1024, distance=rest.Distance.COSINE)})


True

In [90]:
client.create_collection(
    collection_name='samarth111',
    vectors_config={},
    sparse_vectors_config={
        "text": rest.SparseVectorParams(
            index=rest.SparseIndexParams(
                on_disk=False,
            )
        )
    },
)

True

In [248]:
embeddings_dense = OllamaEmbeddings(model="mxbai-embed-large:latest", base_url="http://10.10.64.25:11434") 


In [265]:
embeddings_sparse = sparse_embed()

In [158]:
chunks

[Document(metadata={}, page_content='The late 19th and early 20th centuries were a time of unparalleled transformation. Industrial cities expanded at a rate never seen before. Streets thrummed with life: the clang of tramcars mixed with the clip-clop of horse hooves, while the occasional motorcar roared through the chaos, signaling the future. Factories spewed smoke into the skies, yet the energy of invention seemed to permeate every corner of urban life. People gathered in cafes and public squares to discuss ideas that spanned the mundane and the extraordinary. Philosophers debated morality while engineers sketched diagrams of machines that could communicate across oceans.\n\nScience was experiencing its own revolution. Electricity, once a mysterious phenomenon, was being harnessed to light homes and power machines. Theories of matter and energy were being rewritten; the names of Maxwell, Einstein, and Curie became synonymous with curiosity and genius. In classrooms and laboratories, 

In [159]:
embeddings_sparse.embed_documents(texts)

[{'indices': [859229416,
   1654875254,
   1724426273,
   290563740,
   1147922090,
   2058513491,
   751136919,
   524852419,
   91759785,
   1966288579,
   1997014908,
   1151308235,
   1969182875,
   620220149,
   1421544012,
   1697167928,
   2019785588,
   1786414702,
   738283235,
   1977712632,
   1346960778,
   2004803347,
   90002279,
   1732110808,
   674357371,
   676843137,
   669676499,
   2015064403,
   1904070401,
   573216872,
   1603062076,
   764441889,
   1283721936,
   785651815,
   1117393019,
   1742567733,
   1740045773,
   267423339,
   1363847735,
   1481737067,
   492346370,
   1894123915,
   1300596145,
   2099192796,
   1230453951,
   46395304,
   1315188673,
   441260191,
   2032101475,
   610868587,
   1262801366,
   476627540,
   2030235830,
   2133835312,
   2068085293,
   856081220,
   1998364961,
   1284918442,
   1228389567,
   2100873926,
   1208535885,
   1432619228,
   1852515656,
   1135884324,
   796388130,
   1477368786,
   254310662,
   1223678

In [302]:
docs = embeddings_sparse.embed_documents(texts)
indices = [emb['indices'] for emb in docs]
values = [emb['values'] for emb in docs]

client.upsert(
    collection_name='samarth111',
    points=[
        rest.PointStruct(
            id=1,
            payload={},  # Add any additional payload if necessary
            vector={
                "text": rest.SparseVector(
                    indices=indices[0], values=values[0]
                )
            },
        )
    ],
)

UpdateResult(operation_id=3, status=<UpdateStatus.COMPLETED: 'completed'>)

In [303]:
vct_sparse.add_documents(docs)

AttributeError: 'dict' object has no attribute 'id'

In [ ]:

result = client.search(
    collection_name='samarth111',
    query_vector=rest.NamedSparseVector(
        name="text",
        vector=rest.SparseVector(
            indices=q_ind,
            values=q_val,
        ),
    ),
    with_vectors=True,
    score_threshold=-9999999,
    limit=3
)

C:\Users\samarth.srivastava\AppData\Local\Temp\ipykernel_29468\1738000755.py:1: DeprecationWarning: `search` method is deprecated and will be removed in the future. Use `query_points` instead.
  result = client.search(


In [299]:
from langchain_core.embeddings.embeddings import Embeddings
from fastembed import SparseTextEmbedding
from langchain_core.documents import Document
from qdrant_client.http.models import SparseVector
from qdrant_client.http import models as rest


class sparse_embed(Embeddings):
    embeddings_sparsed = SparseTextEmbedding(model_name="Qdrant/bm25")
    def embed_documents(self, texts):
        embeds = list(self.embeddings_sparsed.embed(texts))
        return [
            {"indices": e.indices.tolist(), "values": e.values.tolist()}
            for e in embeds
        ]
    def embed_query(self, text):
        e = list(self.embeddings_sparsed.embed(text))[0]
        print({"values":e.values.tolist(), "indices":e.indices.tolist()})
        return rest.NamedSparseVector(
                                    name="text",
                                    vector=rest.SparseVector(
                                    indices=e.indices.tolist(),
                                    values=e.values.tolist(),
                                ),
                            )
def get_documents(query):
    get_em = sparse_embed()
    query_vec = get_em.embed_query(query)
    q_ind = query_vec.vector.indices
    q_val = query_vec.vector.values
    result = client.search(
    collection_name='samarth111',
    query_vector=rest.NamedSparseVector(
        name="text",
        vector=rest.SparseVector(
            indices=q_ind,
            values=q_val,
        ),
    ),
    with_vectors=True,
    score_threshold=-9999999,
    limit=3
    )
    return result

Fetching 30 files: 100%|██████████| 30/30 [00:00<?, ?it/s]


In [300]:
get_documents('weaving into the fabric')

{'values': [1.6832038254632398, 1.6832038254632398], 'indices': [611825665, 345572175]}


C:\Users\samarth.srivastava\AppData\Local\Temp\ipykernel_29468\3120311363.py:31: DeprecationWarning: `search` method is deprecated and will be removed in the future. Use `query_points` instead.
  result = client.search(


[]

[ScoredPoint(id=1, version=2, score=6.6941514, payload={}, vector={'text': SparseVector(indices=[7225413, 23584428, 46395304, 74040069, 90002279, 91759785, 91946539, 114620842, 254310662, 267423339, 290563740, 364537710, 412017637, 418093614, 440637894, 441260191, 476627540, 492346370, 524852419, 551536775, 573216872, 581432272, 609270800, 610868587, 620220149, 669676499, 674357371, 676843137, 738283235, 751136919, 764441889, 785651815, 796388130, 856081220, 859229416, 874166076, 939363856, 1020018376, 1034124399, 1038763317, 1117393019, 1135884324, 1147922090, 1150284091, 1151308235, 1179569313, 1181427864, 1205676073, 1208535885, 1223678379, 1228389567, 1230453951, 1262801366, 1283721936, 1284918442, 1300596145, 1315188673, 1346960778, 1363847735, 1421544012, 1426673955, 1427920598, 1432619228, 1477368786, 1481737067, 1539683617, 1603062076, 1654875254, 1667659232, 1697167928, 1724426273, 1732110808, 1740045773, 1742567733, 1786414702, 1828323383, 1852515656, 1894123915, 1895143787, 

In [ ]:
[ScoredPoint(id=1, version=2, score=6.6941514, payload={}, vector={'text': SparseVector(indices=[7225413, 23584428, 46395304, 74040069, 90002279, 91759785, 91946539, 114620842, 254310662, 267423339, 290563740, 364537710, 412017637, 418093614, 440637894, 441260191, 476627540, 492346370, 524852419, 551536775, 573216872, 581432272, 609270800, 610868587, 620220149, 669676499, 674357371, 676843137, 738283235, 751136919, 764441889, 785651815, 796388130, 856081220, 859229416, 874166076, 939363856, 1020018376, 1034124399, 1038763317, 1117393019, 1135884324, 1147922090, 1150284091, 1151308235, 1179569313, 1181427864, 1205676073, 1208535885, 1223678379, 1228389567, 1230453951, 1262801366, 1283721936, 1284918442, 1300596145, 1315188673, 1346960778, 1363847735, 1421544012, 1426673955, 1427920598, 1432619228, 1477368786, 1481737067, 1539683617, 1603062076, 1654875254, 1667659232, 1697167928, 1724426273, 1732110808, 1740045773, 1742567733, 1786414702, 1828323383, 1852515656, 1894123915, 1895143787, 1904070401, 1966288579, 1969182875, 1977712632, 1997014908, 1998364961, 2004803347, 2015064403, 2019785588, 2030235830, 2032101475, 2046950111, 2058513491, 2061075054, 2068085293, 2099192796, 2100873926, 2133835312], values=[1.3292423, 1.3292423, 1.3292423, 1.3292423, 1.3292423, 1.3292423, 1.3292423, 1.3292423, 1.3292423, 1.3292423, 1.3292423, 1.3292423, 1.3292423, 1.3292423, 1.3292423, 1.3292423, 1.3292423, 1.3292423, 1.3292423, 1.3292423, 1.3292423, 1.3292423, 1.3292423, 1.3292423, 1.3292423, 1.3292423, 1.3292423, 1.3292423, 1.3292423, 1.3292423, 1.3292423, 1.3292423, 1.3292423, 1.3292423, 1.3292423, 1.3292423, 1.3292423, 1.3292423, 1.3292423, 1.3292423, 1.3292423, 1.3292423, 1.3292423, 1.3292423, 1.3292423, 1.3292423, 1.3292423, 1.3292423, 1.3292423, 1.3292423, 1.6572016, 1.3292423, 1.3292423, 1.3292423, 1.3292423, 1.3292423, 1.3292423, 1.3292423, 1.3292423, 1.3292423, 1.3292423, 1.3292423, 1.3292423, 1.3292423, 1.3292423, 1.3292423, 1.3292423, 1.3292423, 1.3292423, 1.3292423, 1.3292423, 1.3292423, 1.3292423, 1.6572016, 1.3292423, 1.3292423, 1.3292423, 1.3292423, 1.3292423, 1.3292423, 1.3292423, 1.3292423, 1.3292423, 1.3292423, 1.3292423, 1.3292423, 1.3292423, 1.6572016, 1.3292423, 1.3292423, 1.3292423, 1.3292423, 1.3292423, 1.3292423, 1.3292423, 1.6572016, 1.3292423])}, shard_key=None, order_value=None)]


NameError: name 'ScoredPoint' is not defined

In [266]:
vct_sparse = Qdrant(collection_name='samarth111', client=client, vector_name='text', embeddings=embeddings_sparse)

In [301]:
vct_sparse.from_documents(embedding=embeddings_sparse, documents=result)

AttributeError: 'ScoredPoint' object has no attribute 'page_content'

In [267]:
r = vct_sparse.as_retriever()

In [ ]:
vct_sparse.from_documents()

In [ ]:
query = 'electricity, once a mysterious phenomenon'
q_em = embeddings_sparse.embed_query(query)


{'values': [1.6786885245901642, 1.6786885245901642, 1.6786885245901642], 'indices': [254310662, 1223678379, 7225413]}


In [278]:
texts[1]

' classrooms and laboratories, young minds explored principles that would shape the modern world: radio waves that could transmit voices without wires, chemical reactions that defied conventional expectations, and the strange behaviors of particles invisible to the naked eye. Each discovery was a thread weaving into the fabric of what would one day be called modern science.\n\nAt the same time, society grappled with the consequences of progress. Urbanization brought not only opportunity but also congestion, disease, and inequality. The streets were alive with vendors, laborers, artists, and children, each negotiating a cityscape that was expanding faster than its infrastructure. Public transportation emerged as a solution, yet it could not fully tame the chaos. Social reformers, activists, and writers chronicled these changes, advocating for public health, education, and workers’ rights. Literature became a mirror to society, reflecting both the beauty and the tension of a world on the

In [279]:
print(" classrooms and laboratories, young minds explored principles that would shape the modern world: radio waves that could transmit voices without wires, chemical reactions that defied conventional expectations, and the strange behaviors of particles invisible to the naked eye. Each discovery was a thread weaving into the fabric of what would one day be called modern science.\n\nAt the same time, society grappled with the consequences of progress. Urbanization brought not only opportunity but also congestion, disease, and inequality. The streets were alive with vendors, laborers, artists, and children, each negotiating a cityscape that was expanding faster than its infrastructure. Public transportation emerged as a solution, yet it could not fully tame the chaos. Social reformers, activists, and writers chronicled these changes, advocating for public health, education, and workers’ rights. Literature became a mirror to society, reflecting both the beauty and the tension of a world on the brink of modernity.\n\nTravel and exploration captured the imagination of many. Steamships")

 classrooms and laboratories, young minds explored principles that would shape the modern world: radio waves that could transmit voices without wires, chemical reactions that defied conventional expectations, and the strange behaviors of particles invisible to the naked eye. Each discovery was a thread weaving into the fabric of what would one day be called modern science.

At the same time, society grappled with the consequences of progress. Urbanization brought not only opportunity but also congestion, disease, and inequality. The streets were alive with vendors, laborers, artists, and children, each negotiating a cityscape that was expanding faster than its infrastructure. Public transportation emerged as a solution, yet it could not fully tame the chaos. Social reformers, activists, and writers chronicled these changes, advocating for public health, education, and workers’ rights. Literature became a mirror to society, reflecting both the beauty and the tension of a world on the br

In [274]:
q_ind = q_em.vector.indices
q_val = q_em.vector.values

In [268]:
r.invoke('klajdfa lakjsdf lskdjf')

{'values': [1.6786885245901642, 1.6786885245901642, 1.6786885245901642], 'indices': [544586508, 1104477081, 410499446]}


ValidationError: 2 validation errors for NamedVector
vector.0
  Input should be a valid number [type=float_type, input_value=('name', 'text'), input_type=tuple]
    For further information visit https://errors.pydantic.dev/2.11/v/float_type
vector.1
  Input should be a valid number [type=float_type, input_value=('vector', SparseVector(i...2, 1.6786885245901642])), input_type=tuple]
    For further information visit https://errors.pydantic.dev/2.11/v/float_type

In [ ]:
docs = embeddings_sparse.embed_documents(texts)
indices = [emb['indices'] for emb in docs]
values = [emb['values'] for emb in docs]


AttributeError: 'list' object has no attribute 'indices'

In [173]:
vct_sparse.similarity_search('what is this', k=3)

what is this [] []


ValidationError: 1 validation error for NamedVector
vector.0
  Input should be a valid number [type=float_type, input_value={'indices': [], 'values': []}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.11/v/float_type

In [64]:
indices

[[859229416,
  1654875254,
  1724426273,
  290563740,
  1147922090,
  2058513491,
  751136919,
  524852419,
  91759785,
  1966288579,
  1997014908,
  1151308235,
  1969182875,
  620220149,
  1421544012,
  1697167928,
  2019785588,
  1786414702,
  738283235,
  1977712632,
  1346960778,
  2004803347,
  90002279,
  1732110808,
  674357371,
  676843137,
  669676499,
  2015064403,
  1904070401,
  573216872,
  1603062076,
  764441889,
  1283721936,
  785651815,
  1117393019,
  1742567733,
  1740045773,
  267423339,
  1363847735,
  1481737067,
  492346370,
  1894123915,
  1300596145,
  2099192796,
  1230453951,
  46395304,
  1315188673,
  441260191,
  2032101475,
  610868587,
  1262801366,
  476627540,
  2030235830,
  2133835312,
  2068085293,
  856081220,
  1998364961,
  1284918442,
  1228389567,
  2100873926,
  1208535885,
  1432619228,
  1852515656,
  1135884324,
  796388130,
  1477368786,
  254310662,
  1223678379,
  7225413,
  1426673955,
  1539683617,
  1150284091,
  2061075054,
  41201

In [31]:
model1 = embeddings_dense
model2 = embeddings_sparse

In [ ]:
vct = Qdrant(client=client, collection_name='samarth101', vector_name='some_name', embeddings=embeddings_sparse)

In [40]:
ret = vct.as_retriever()

In [53]:
text = '''The late 19th and early 20th centuries were a time of unparalleled transformation. Industrial cities expanded at a rate never seen before. Streets thrummed with life: the clang of tramcars mixed with the clip-clop of horse hooves, while the occasional motorcar roared through the chaos, signaling the future. Factories spewed smoke into the skies, yet the energy of invention seemed to permeate every corner of urban life. People gathered in cafes and public squares to discuss ideas that spanned the mundane and the extraordinary. Philosophers debated morality while engineers sketched diagrams of machines that could communicate across oceans.

Science was experiencing its own revolution. Electricity, once a mysterious phenomenon, was being harnessed to light homes and power machines. Theories of matter and energy were being rewritten; the names of Maxwell, Einstein, and Curie became synonymous with curiosity and genius. In classrooms and laboratories, young minds explored principles that would shape the modern world: radio waves that could transmit voices without wires, chemical reactions that defied conventional expectations, and the strange behaviors of particles invisible to the naked eye. Each discovery was a thread weaving into the fabric of what would one day be called modern science.

At the same time, society grappled with the consequences of progress. Urbanization brought not only opportunity but also congestion, disease, and inequality. The streets were alive with vendors, laborers, artists, and children, each negotiating a cityscape that was expanding faster than its infrastructure. Public transportation emerged as a solution, yet it could not fully tame the chaos. Social reformers, activists, and writers chronicled these changes, advocating for public health, education, and workers’ rights. Literature became a mirror to society, reflecting both the beauty and the tension of a world on the brink of modernity.

Travel and exploration captured the imagination of many. Steamships crossed oceans faster than ever, bringing distant cultures into closer contact. Expeditions to uncharted territories, from the frozen expanses of Antarctica to the dense rainforests of the Amazon, yielded stories of peril and wonder. Photographs and journals documented these adventures, providing glimpses into lives far removed from the everyday urban experience. Maps were redrawn, new species cataloged, and the understanding of geography and biodiversity expanded exponentially.

Meanwhile, the world of ideas was not confined to laboratories or lecture halls. Philosophy, art, and literature intertwined with science and politics to shape public discourse. Writers experimented with narrative forms, painters with perspective and abstraction, and composers with the boundaries of harmony. Newspapers and periodicals proliferated, spreading knowledge and opinion, and making debates accessible to a broader audience than ever before. Literacy rates climbed, education expanded, and the appetite for knowledge seemed insatiable. Libraries became sanctuaries of discovery, their shelves filled with books that connected past, present, and future thinkers.

Technology, perhaps more than any other force, was reshaping daily life. The telegraph had already revolutionized communication; the telephone promised to bring voices instantly across great distances. Cameras captured moments that were once ephemeral, and moving pictures began to entertain and inform. Scientific instruments allowed measurements with precision unimaginable a generation earlier. Everyday objects, from the lightbulb to the typewriter, transformed the way people worked, learned, and socialized. Even simple curiosity could now lead to rapid dissemination of knowledge, and the acceleration of ideas felt dizzying.

Yet, amidst this progress, questions of ethics and consequence loomed. Industrialization brought environmental degradation, and the relentless pursuit of profit often clashed with the welfare of workers and communities. Philosophers and scientists alike pondered the responsibilities that came with newfound power. Could society balance growth with humanity? Could invention coexist with justice? These questions, while timeless, were amplified by the speed of change and the scope of human ambition.

Cultural exchange blossomed as the world became more interconnected. Artifacts from distant lands inspired design, storytelling borrowed motifs across borders, and ideas traveled faster than armies or merchants. Music incorporated instruments and styles from continents apart; literature reflected voices previously unheard; fashion merged tradition with innovation. Cosmopolitan centers emerged, hubs of trade, creativity, and discourse that drew people from every corner of the globe. The notion of a truly global society was beginning to take shape, though its implications were still only partially understood.

Looking back from a modern perspective, these decades were more than just a series of inventions and social changes—they were the crucible in which the modern world was forged. Each experiment, each philosophical debate, each architectural blueprint contributed to a vast network of human knowledge and capability. While the individual pieces might seem disparate, together they reveal a tapestry of innovation, resilience, and imagination. The people of this era, both ordinary and extraordinary, laid foundations that continue to influence science, society, and culture to this day.'''

In [54]:
from chunking.recursive_char import RecursiveChunker
chunker = RecursiveChunker()
chunks = chunker.create_chunks(texts=[text])

In [24]:
for i in (0, len(chunks)-1):
    chunks[i].id = i

In [21]:
for i in (1, 2):
    print(i)

1
2


In [59]:
texts = [chunk.page_content for chunk in chunks]

In [ ]:
from langchain_redis import RedisVectorStore, RedisConfig
from langchain_ollama import OllamaEmbeddings
# Configure Redis with HNSW for ANN search
config = RedisConfig(
    index_name="rag_documents",
    redis_url="redis://localhost:6379",
    indexing_algorithm="HNSW",  # Enables ANN search
    distance_metric="COSINE",
    vector_datatype="FLOAT32",
    metadata_schema=[
        {"name": "source", "type": "tag"},
        {"name": "page", "type": "numeric"}
    ]
)


In [2]:
embeddings = OllamaEmbeddings(model="mxbai-embed-large:latest", base_url="http://10.10.64.25:11434") 

In [ ]:
vector_store = RedisVectorStore(redis_url="http://127.0.0.1:6379", embeddings=embeddings, config=config)


14:31:18 httpx INFO   HTTP Request: POST http://10.10.64.25:11434/api/embed "HTTP/1.1 200 OK"


In [12]:
vector_store.add_texts(
        texts=["This is a document about AI.", "Another document on machine learning."],
        metadatas=[{"source": "doc1"}, {"source": "doc2"}]
    )


14:45:26 httpx INFO   HTTP Request: POST http://10.10.64.25:11434/api/embed "HTTP/1.1 200 OK"


['rag_documents:01K6WCVTWQ3RAFX7ZNMPKWZ035',
 'rag_documents:01K6WCVTWQ3RAFX7ZNMPKWZ036']

In [13]:
ret = vector_store.as_retriever()

In [14]:
ret.invoke('what is ai')

14:45:54 httpx INFO   HTTP Request: POST http://10.10.64.25:11434/api/embed "HTTP/1.1 200 OK"


[Document(metadata={'source': 'doc1'}, page_content='This is a document about AI.'),
 Document(metadata={'source': 'doc1'}, page_content='This is a document about AI.'),
 Document(metadata={'source': 'doc2'}, page_content='Another document on machine learning.'),
 Document(metadata={'source': 'doc2'}, page_content='Another document on machine learning.')]

In [11]:
query = "What is AI?"
results = vector_store.similarity_search(query, k=2)
results

14:31:36 httpx INFO   HTTP Request: POST http://10.10.64.25:11434/api/embed "HTTP/1.1 200 OK"


[Document(metadata={'source': 'doc1'}, page_content='This is a document about AI.'),
 Document(metadata={'source': 'doc2'}, page_content='Another document on machine learning.')]

In [1]:
from pymilvus import Collection, MilvusException, connections, db, utility

conn = connections.connect(host="127.0.0.1", port=19530)

In [2]:
db.list_database()

['default']

In [15]:
from dense_retrieval import DenseRetrieval

In [16]:
ob = DenseRetrieval(collection_name = 'samarth2', user_name = 'common')

In [17]:
text = '''The late 19th and early 20th centuries were a time of unparalleled transformation. Industrial cities expanded at a rate never seen before. Streets thrummed with life: the clang of tramcars mixed with the clip-clop of horse hooves, while the occasional motorcar roared through the chaos, signaling the future. Factories spewed smoke into the skies, yet the energy of invention seemed to permeate every corner of urban life. People gathered in cafes and public squares to discuss ideas that spanned the mundane and the extraordinary. Philosophers debated morality while engineers sketched diagrams of machines that could communicate across oceans.

Science was experiencing its own revolution. Electricity, once a mysterious phenomenon, was being harnessed to light homes and power machines. Theories of matter and energy were being rewritten; the names of Maxwell, Einstein, and Curie became synonymous with curiosity and genius. In classrooms and laboratories, young minds explored principles that would shape the modern world: radio waves that could transmit voices without wires, chemical reactions that defied conventional expectations, and the strange behaviors of particles invisible to the naked eye. Each discovery was a thread weaving into the fabric of what would one day be called modern science.

At the same time, society grappled with the consequences of progress. Urbanization brought not only opportunity but also congestion, disease, and inequality. The streets were alive with vendors, laborers, artists, and children, each negotiating a cityscape that was expanding faster than its infrastructure. Public transportation emerged as a solution, yet it could not fully tame the chaos. Social reformers, activists, and writers chronicled these changes, advocating for public health, education, and workers’ rights. Literature became a mirror to society, reflecting both the beauty and the tension of a world on the brink of modernity.

Travel and exploration captured the imagination of many. Steamships crossed oceans faster than ever, bringing distant cultures into closer contact. Expeditions to uncharted territories, from the frozen expanses of Antarctica to the dense rainforests of the Amazon, yielded stories of peril and wonder. Photographs and journals documented these adventures, providing glimpses into lives far removed from the everyday urban experience. Maps were redrawn, new species cataloged, and the understanding of geography and biodiversity expanded exponentially.

Meanwhile, the world of ideas was not confined to laboratories or lecture halls. Philosophy, art, and literature intertwined with science and politics to shape public discourse. Writers experimented with narrative forms, painters with perspective and abstraction, and composers with the boundaries of harmony. Newspapers and periodicals proliferated, spreading knowledge and opinion, and making debates accessible to a broader audience than ever before. Literacy rates climbed, education expanded, and the appetite for knowledge seemed insatiable. Libraries became sanctuaries of discovery, their shelves filled with books that connected past, present, and future thinkers.

Technology, perhaps more than any other force, was reshaping daily life. The telegraph had already revolutionized communication; the telephone promised to bring voices instantly across great distances. Cameras captured moments that were once ephemeral, and moving pictures began to entertain and inform. Scientific instruments allowed measurements with precision unimaginable a generation earlier. Everyday objects, from the lightbulb to the typewriter, transformed the way people worked, learned, and socialized. Even simple curiosity could now lead to rapid dissemination of knowledge, and the acceleration of ideas felt dizzying.

Yet, amidst this progress, questions of ethics and consequence loomed. Industrialization brought environmental degradation, and the relentless pursuit of profit often clashed with the welfare of workers and communities. Philosophers and scientists alike pondered the responsibilities that came with newfound power. Could society balance growth with humanity? Could invention coexist with justice? These questions, while timeless, were amplified by the speed of change and the scope of human ambition.

Cultural exchange blossomed as the world became more interconnected. Artifacts from distant lands inspired design, storytelling borrowed motifs across borders, and ideas traveled faster than armies or merchants. Music incorporated instruments and styles from continents apart; literature reflected voices previously unheard; fashion merged tradition with innovation. Cosmopolitan centers emerged, hubs of trade, creativity, and discourse that drew people from every corner of the globe. The notion of a truly global society was beginning to take shape, though its implications were still only partially understood.

Looking back from a modern perspective, these decades were more than just a series of inventions and social changes—they were the crucible in which the modern world was forged. Each experiment, each philosophical debate, each architectural blueprint contributed to a vast network of human knowledge and capability. While the individual pieces might seem disparate, together they reveal a tapestry of innovation, resilience, and imagination. The people of this era, both ordinary and extraordinary, laid foundations that continue to influence science, society, and culture to this day.'''


In [18]:
check = ob.get_retriever(raw_text = text)

The ids parameter is ignored when auto_id is True. The ids will be generated automatically.


In [19]:
check.invoke('Connection and mixing of traditions')

[Document(metadata={'pk': 461372690834134784}, page_content=' environmental degradation, and the relentless pursuit of profit often clashed with the welfare of workers and communities. Philosophers and scientists alike pondered the responsibilities that came with newfound power. Could society balance growth with humanity? Could invention coexist with justice? These questions, while timeless, were amplified by the speed of change and the scope of human ambition.\n\nCultural exchange blossomed as the world became more interconnected. Artifacts from distant lands inspired design, storytelling borrowed motifs across borders, and ideas traveled faster than armies or merchants. Music incorporated instruments and styles from continents apart; literature reflected voices previously unheard; fashion merged tradition with innovation. Cosmopolitan centers emerged, hubs of trade, creativity, and discourse that drew people from every corner of the globe. The notion of a truly global society was beg

In [3]:
db.create_database('samarth')

In [4]:
db.list_database()

['default', 'samarth']

In [9]:
db.using_database('default')

In [25]:
!pip install pdfplumber

   ---------------------------------------- 0.0/5.6 MB ? eta -:--:--
   -------------------- ------------------- 2.9/5.6 MB 16.8 MB/s eta 0:00:01
   ---------------------------------------- 5.6/5.6 MB 16.3 MB/s eta 0:00:00
   ---------------------------------------- 0.0/2.9 MB ? eta -:--:--
   ---------------------------------------- 2.9/2.9 MB 21.0 MB/s eta 0:00:00


In [32]:
import pdfplumber
text = ''
with pdfplumber.open(r"C:\Users\samarth.srivastava\Downloads\test_doc.pdf") as pdf:
    
    for page in pdf.pages:
        text = text+page.extract_text()
        # print(text)
        # break

In [37]:
txt1 = text.replace(text2, '')

In [40]:
from langchain_qdrant import Qdrant
from qdrant_client import QdrantClient
from langchain_ollama import OllamaEmbeddings
from qdrant_client.http import models as rest
from Retrievers.SparseEmbeddings import sparse_embed
client = QdrantClient(url='http://localhost:6333')

Fetching 30 files:   0%|          | 0/30 [00:00<?, ?it/s]c:\Users\samarth.srivastava\Desktop\new_check_proj\test_env2\lib\site-packages\huggingface_hub\file_download.py:140: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\samarth.srivastava\AppData\Local\Temp\fastembed_cache\models--Qdrant--bm25. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Fetching 30 fi

In [49]:
vector_name = 'new'
client.create_collection(collection_name='sam', vectors_config={}, sparse_vectors_config={vector_name: rest.SparseVectorParams(index=rest.SparseIndexParams(on_disk=False,))},)
# client.collection_exists(cl)

UnexpectedResponse: Unexpected Response: 409 (Conflict)
Raw response content:
b'{"status":{"error":"Wrong input: Collection `sam` already exists!"},"time":0.000832908}'

In [44]:
client.collection_exists(collection_name = 'sam1')

False

In [48]:
ret = client.get_collection(collection_name = 'sam')

In [38]:
print(txt1)

Providedproperattributionisprovided,Googleherebygrantspermissionto
reproducethetablesandfiguresinthispapersolelyforuseinjournalisticor
scholarlyworks.
Attention Is All You Need
AshishVaswani∗ NoamShazeer∗ NikiParmar∗ JakobUszkoreit∗
GoogleBrain GoogleBrain GoogleResearch GoogleResearch
avaswani@google.com noam@google.com nikip@google.com usz@google.com
LlionJones∗ AidanN.Gomez∗ † ŁukaszKaiser∗
GoogleResearch UniversityofToronto GoogleBrain
llion@google.com aidan@cs.toronto.edu lukaszkaiser@google.com
IlliaPolosukhin∗ ‡
illia.polosukhin@gmail.com
Abstract
Thedominantsequencetransductionmodelsarebasedoncomplexrecurrentor
convolutionalneuralnetworksthatincludeanencoderandadecoder. Thebest
performing models also connect the encoder and decoder through an attention
mechanism. We propose a new simple network architecture, the Transformer,
basedsolelyonattentionmechanisms,dispensingwithrecurrenceandconvolutions
entirely. Experiments on two machine translation tasks show these models to
besupe

In [8]:
utility.list_collections()

['langchain_example']

In [1]:
raw_text = '''Providedproperattributionisprovided,Googleherebygrantspermissionto
reproducethetablesandfiguresinthispapersolelyforuseinjournalisticor
scholarlyworks.
Attention Is All You Need
AshishVaswani∗ NoamShazeer∗ NikiParmar∗ JakobUszkoreit∗
GoogleBrain GoogleBrain GoogleResearch GoogleResearch
avaswani@google.com noam@google.com nikip@google.com usz@google.com
LlionJones∗ AidanN.Gomez∗ † ŁukaszKaiser∗
GoogleResearch UniversityofToronto GoogleBrain
llion@google.com aidan@cs.toronto.edu lukaszkaiser@google.com
IlliaPolosukhin∗ ‡
illia.polosukhin@gmail.com
Abstract
Thedominantsequencetransductionmodelsarebasedoncomplexrecurrentor
convolutionalneuralnetworksthatincludeanencoderandadecoder. Thebest
performing models also connect the encoder and decoder through an attention
mechanism. We propose a new simple network architecture, the Transformer,
basedsolelyonattentionmechanisms,dispensingwithrecurrenceandconvolutions
entirely. Experiments on two machine translation tasks show these models to
besuperiorinqualitywhilebeingmoreparallelizableandrequiringsignificantly
less time to train. Our model achieves 28.4 BLEU on the WMT 2014 English-
to-German translation task, improving over the existing best results, including
ensembles,byover2BLEU.OntheWMT2014English-to-Frenchtranslationtask,
ourmodelestablishesanewsingle-modelstate-of-the-artBLEUscoreof41.8after
trainingfor3.5daysoneightGPUs,asmallfractionofthetrainingcostsofthe
bestmodelsfromtheliterature. WeshowthattheTransformergeneralizeswellto
othertasksbyapplyingitsuccessfullytoEnglishconstituencyparsingbothwith
largeandlimitedtrainingdata.
∗Equalcontribution.Listingorderisrandom.JakobproposedreplacingRNNswithself-attentionandstarted
theefforttoevaluatethisidea.Ashish,withIllia,designedandimplementedthefirstTransformermodelsand
hasbeencruciallyinvolvedineveryaspectofthiswork.Noamproposedscaleddot-productattention,multi-head
attentionandtheparameter-freepositionrepresentationandbecametheotherpersoninvolvedinnearlyevery
detail.Nikidesigned,implemented,tunedandevaluatedcountlessmodelvariantsinouroriginalcodebaseand
tensor2tensor.Llionalsoexperimentedwithnovelmodelvariants,wasresponsibleforourinitialcodebase,and
efficientinferenceandvisualizations.LukaszandAidanspentcountlesslongdaysdesigningvariouspartsofand
implementingtensor2tensor,replacingourearliercodebase,greatlyimprovingresultsandmassivelyaccelerating
ourresearch.
†WorkperformedwhileatGoogleBrain.
‡WorkperformedwhileatGoogleResearch.
31stConferenceonNeuralInformationProcessingSystems(NIPS2017),LongBeach,CA,USA.
3202
guA
2
]LC.sc[
7v26730.6071:viXra1 Introduction
Recurrentneuralnetworks,longshort-termmemory[13]andgatedrecurrent[7]neuralnetworks
inparticular,havebeenfirmlyestablishedasstateoftheartapproachesinsequencemodelingand
transductionproblemssuchaslanguagemodelingandmachinetranslation[35,2,5]. Numerous
effortshavesincecontinuedtopushtheboundariesofrecurrentlanguagemodelsandencoder-decoder
architectures[38,24,15].
Recurrentmodelstypicallyfactorcomputationalongthesymbolpositionsoftheinputandoutput
sequences. Aligningthepositionstostepsincomputationtime,theygenerateasequenceofhidden
statesh ,asafunctionoftheprevioushiddenstateh andtheinputforpositiont. Thisinherently
t t−1
sequentialnatureprecludesparallelizationwithintrainingexamples,whichbecomescriticalatlonger
sequencelengths,asmemoryconstraintslimitbatchingacrossexamples. Recentworkhasachieved
significantimprovementsincomputationalefficiencythroughfactorizationtricks[21]andconditional
computation[32],whilealsoimprovingmodelperformanceincaseofthelatter. Thefundamental
constraintofsequentialcomputation,however,remains.
Attentionmechanismshavebecomeanintegralpartofcompellingsequencemodelingandtransduc-
tionmodelsinvarioustasks,allowingmodelingofdependencieswithoutregardtotheirdistancein
theinputoroutputsequences[2,19]. Inallbutafewcases[27],however,suchattentionmechanisms
areusedinconjunctionwitharecurrentnetwork.
InthisworkweproposetheTransformer,amodelarchitectureeschewingrecurrenceandinstead
relyingentirelyonanattentionmechanismtodrawglobaldependenciesbetweeninputandoutput.
TheTransformerallowsforsignificantlymoreparallelizationandcanreachanewstateoftheartin
translationqualityafterbeingtrainedforaslittleastwelvehoursoneightP100GPUs.
2 Background
ThegoalofreducingsequentialcomputationalsoformsthefoundationoftheExtendedNeuralGPU
[16],ByteNet[18]andConvS2S[9],allofwhichuseconvolutionalneuralnetworksasbasicbuilding
block,computinghiddenrepresentationsinparallelforallinputandoutputpositions.Inthesemodels,
thenumberofoperationsrequiredtorelatesignalsfromtwoarbitraryinputoroutputpositionsgrows
inthedistancebetweenpositions,linearlyforConvS2SandlogarithmicallyforByteNet. Thismakes
it more difficult to learn dependencies between distant positions [12]. In the Transformer this is
reducedtoaconstantnumberofoperations, albeitatthecostofreducedeffectiveresolutiondue
to averaging attention-weighted positions, an effect we counteract with Multi-Head Attention as
describedinsection3.2.
Self-attention,sometimescalledintra-attentionisanattentionmechanismrelatingdifferentpositions
ofasinglesequenceinordertocomputearepresentationofthesequence. Self-attentionhasbeen
usedsuccessfullyinavarietyoftasksincludingreadingcomprehension,abstractivesummarization,
textualentailmentandlearningtask-independentsentencerepresentations[4,27,28,22].
End-to-endmemorynetworksarebasedonarecurrentattentionmechanisminsteadofsequence-
alignedrecurrenceandhavebeenshowntoperformwellonsimple-languagequestionansweringand
languagemodelingtasks[34].
To the best of our knowledge, however, the Transformer is the first transduction model relying
entirelyonself-attentiontocomputerepresentationsofitsinputandoutputwithoutusingsequence-
alignedRNNsorconvolution. Inthefollowingsections,wewilldescribetheTransformer,motivate
self-attentionanddiscussitsadvantagesovermodelssuchas[17,18]and[9].
3 ModelArchitecture
Mostcompetitiveneuralsequencetransductionmodelshaveanencoder-decoderstructure[5,2,35].
Here, the encoder maps an input sequence of symbol representations (x ,...,x ) to a sequence
1 n
of continuous representations z = (z ,...,z ). Given z, the decoder then generates an output
1 n
sequence(y ,...,y )ofsymbolsoneelementatatime. Ateachstepthemodelisauto-regressive
1 m
[10],consumingthepreviouslygeneratedsymbolsasadditionalinputwhengeneratingthenext.
2Figure1: TheTransformer-modelarchitecture.
TheTransformerfollowsthisoverallarchitectureusingstackedself-attentionandpoint-wise,fully
connectedlayersforboththeencoderanddecoder,shownintheleftandrighthalvesofFigure1,
respectively.
3.1 EncoderandDecoderStacks
Encoder: The encoder is composed of a stack of N = 6 identical layers. Each layer has two
sub-layers. Thefirstisamulti-headself-attentionmechanism,andthesecondisasimple,position-
wisefullyconnectedfeed-forwardnetwork. Weemployaresidualconnection[11]aroundeachof
the two sub-layers, followed by layer normalization [1]. That is, the output of each sub-layer is
LayerNorm(x+Sublayer(x)),whereSublayer(x)isthefunctionimplementedbythesub-layer
itself. Tofacilitatetheseresidualconnections,allsub-layersinthemodel,aswellastheembedding
layers,produceoutputsofdimensiond =512.
model
Decoder: ThedecoderisalsocomposedofastackofN =6identicallayers. Inadditiontothetwo
sub-layersineachencoderlayer,thedecoderinsertsathirdsub-layer,whichperformsmulti-head
attentionovertheoutputoftheencoderstack. Similartotheencoder,weemployresidualconnections
aroundeachofthesub-layers,followedbylayernormalization. Wealsomodifytheself-attention
sub-layer in the decoder stack to prevent positions from attending to subsequent positions. This
masking,combinedwithfactthattheoutputembeddingsareoffsetbyoneposition,ensuresthatthe
predictionsforpositionicandependonlyontheknownoutputsatpositionslessthani.
3.2 Attention
Anattentionfunctioncanbedescribedasmappingaqueryandasetofkey-valuepairstoanoutput,
wherethequery,keys,values,andoutputareallvectors. Theoutputiscomputedasaweightedsum
3ScaledDot-ProductAttention Multi-HeadAttention
Figure 2: (left) Scaled Dot-Product Attention. (right) Multi-Head Attention consists of several
attentionlayersrunninginparallel.
ofthevalues,wheretheweightassignedtoeachvalueiscomputedbyacompatibilityfunctionofthe
querywiththecorrespondingkey.
3.2.1 ScaledDot-ProductAttention
Wecallourparticularattention"ScaledDot-ProductAttention"(Figure2). Theinputconsistsof
queriesandkeysofdimensiond
k
,a√ndvaluesofdimensiond
v
. Wecomputethedotproductsofthe
querywithallkeys,divideeachby d ,andapplyasoftmaxfunctiontoobtaintheweightsonthe
k
values.
Inpractice,wecomputetheattentionfunctiononasetofqueriessimultaneously,packedtogether
intoamatrixQ. ThekeysandvaluesarealsopackedtogetherintomatricesK andV. Wecompute
thematrixofoutputsas:
QKT
Attention(Q,K,V)=softmax( √ )V (1)
d
k
Thetwomostcommonlyusedattentionfunctionsareadditiveattention[2],anddot-product(multi-
plicative)attention. Dot-productattentionisidenticaltoouralgorithm,exceptforthescalingfactor
of √1 . Additiveattentioncomputesthecompatibilityfunctionusingafeed-forwardnetworkwith
dk
asinglehiddenlayer. Whilethetwoaresimilarintheoreticalcomplexity,dot-productattentionis
muchfasterandmorespace-efficientinpractice,sinceitcanbeimplementedusinghighlyoptimized
matrixmultiplicationcode.
Whileforsmallvaluesofd thetwomechanismsperformsimilarly,additiveattentionoutperforms
k
dotproductattentionwithoutscalingforlargervaluesofd [3]. Wesuspectthatforlargevaluesof
k
d ,thedotproductsgrowlargeinmagnitude,pushingthesoftmaxfunctionintoregionswhereithas
k
extremelysmallgradients4. Tocounteractthiseffect,wescalethedotproductsby √1 .
dk
3.2.2 Multi-HeadAttention
Insteadofperformingasingleattentionfunctionwithd -dimensionalkeys,valuesandqueries,
model
wefounditbeneficialtolinearlyprojectthequeries,keysandvalueshtimeswithdifferent,learned
linearprojectionstod ,d andd dimensions,respectively. Oneachoftheseprojectedversionsof
k k v
queries,keysandvalueswethenperformtheattentionfunctioninparallel,yieldingd -dimensional
v
4Toillustratewhythedotproductsgetlarge,assumethatthecomponentsofqandkareindependentrandom
variableswithmean0andvariance1.Thentheirdotproduct,q·k=
(cid:80)dk
q k ,hasmean0andvarianced .
i=1 i i k
4output values. These are concatenated and once again projected, resulting in the final values, as
depictedinFigure2.
Multi-headattentionallowsthemodeltojointlyattendtoinformationfromdifferentrepresentation
subspacesatdifferentpositions. Withasingleattentionhead,averaginginhibitsthis.
MultiHead(Q,K,V)=Concat(head ,...,head )WO
1 h
wherehead =Attention(QWQ,KWK,VWV)
i i i i
WheretheprojectionsareparametermatricesWQ ∈Rdmodel×dk,WK ∈Rdmodel×dk,WV ∈Rdmodel×dv
i i i
andWO ∈Rhdv×dmodel.
In this work we employ h = 8 parallel attention layers, or heads. For each of these we use
d =d =d /h=64. Duetothereduceddimensionofeachhead,thetotalcomputationalcost
k v model
issimilartothatofsingle-headattentionwithfulldimensionality.
3.2.3 ApplicationsofAttentioninourModel
TheTransformerusesmulti-headattentioninthreedifferentways:
• In"encoder-decoderattention"layers,thequeriescomefromthepreviousdecoderlayer,
andthememorykeysandvaluescomefromtheoutputoftheencoder. Thisallowsevery
positioninthedecodertoattendoverallpositionsintheinputsequence. Thismimicsthe
typical encoder-decoder attention mechanisms in sequence-to-sequence models such as
[38,2,9].
• Theencodercontainsself-attentionlayers. Inaself-attentionlayerallofthekeys,values
andqueriescomefromthesameplace,inthiscase,theoutputofthepreviouslayerinthe
encoder. Eachpositionintheencodercanattendtoallpositionsinthepreviouslayerofthe
encoder.
• Similarly,self-attentionlayersinthedecoderalloweachpositioninthedecodertoattendto
allpositionsinthedecoderuptoandincludingthatposition. Weneedtopreventleftward
informationflowinthedecodertopreservetheauto-regressiveproperty. Weimplementthis
insideofscaleddot-productattentionbymaskingout(settingto−∞)allvaluesintheinput
ofthesoftmaxwhichcorrespondtoillegalconnections. SeeFigure2.
3.3 Position-wiseFeed-ForwardNetworks
Inadditiontoattentionsub-layers,eachofthelayersinourencoderanddecodercontainsafully
connectedfeed-forwardnetwork,whichisappliedtoeachpositionseparatelyandidentically. This
consistsoftwolineartransformationswithaReLUactivationinbetween.
FFN(x)=max(0,xW +b )W +b (2)
1 1 2 2
Whilethelineartransformationsarethesameacrossdifferentpositions,theyusedifferentparameters
from layer to layer. Another way of describing this is as two convolutions with kernel size 1.
The dimensionality of input and output is d = 512, and the inner-layer has dimensionality
model
d =2048.
ff
3.4 EmbeddingsandSoftmax
Similarlytoothersequencetransductionmodels,weuselearnedembeddingstoconverttheinput
tokensandoutputtokenstovectorsofdimensiond . Wealsousetheusuallearnedlineartransfor-
model
mationandsoftmaxfunctiontoconvertthedecoderoutputtopredictednext-tokenprobabilities. In
ourmodel,wesharethesameweightmatrixbetweenthetwoembeddinglayersandthepre-√softmax
lineartransformation,similarto[30]. Intheembeddinglayers,wemultiplythoseweightsby d .
model
5Table1: Maximumpathlengths,per-layercomplexityandminimumnumberofsequentialoperations
fordifferentlayertypes. nisthesequencelength,distherepresentationdimension,kisthekernel
sizeofconvolutionsandrthesizeoftheneighborhoodinrestrictedself-attention.
LayerType ComplexityperLayer Sequential MaximumPathLength
Operations
Self-Attention O(n2·d) O(1) O(1)
Recurrent O(n·d2) O(n) O(n)
Convolutional O(k·n·d2) O(1) O(log (n))
k
Self-Attention(restricted) O(r·n·d) O(1) O(n/r)
3.5 PositionalEncoding
Sinceourmodelcontainsnorecurrenceandnoconvolution,inorderforthemodeltomakeuseofthe
orderofthesequence,wemustinjectsomeinformationabouttherelativeorabsolutepositionofthe
tokensinthesequence. Tothisend,weadd"positionalencodings"totheinputembeddingsatthe
bottomsoftheencoderanddecoderstacks. Thepositionalencodingshavethesamedimensiond
model
astheembeddings,sothatthetwocanbesummed. Therearemanychoicesofpositionalencodings,
learnedandfixed[9].
Inthiswork,weusesineandcosinefunctionsofdifferentfrequencies:
PE =sin(pos/100002i/dmodel)
(pos,2i)
PE =cos(pos/100002i/dmodel)
(pos,2i+1)
whereposisthepositionandiisthedimension. Thatis,eachdimensionofthepositionalencoding
correspondstoasinusoid. Thewavelengthsformageometricprogressionfrom2πto10000·2π. We
chosethisfunctionbecausewehypothesizeditwouldallowthemodeltoeasilylearntoattendby
relativepositions,sinceforanyfixedoffsetk,PE canberepresentedasalinearfunctionof
pos+k
PE .
pos
Wealsoexperimentedwithusinglearnedpositionalembeddings[9]instead,andfoundthatthetwo
versionsproducednearlyidenticalresults(seeTable3row(E)).Wechosethesinusoidalversion
becauseitmayallowthemodeltoextrapolatetosequencelengthslongerthantheonesencountered
duringtraining.
4 WhySelf-Attention
In this section we compare various aspects of self-attention layers to the recurrent and convolu-
tionallayerscommonlyusedformappingonevariable-lengthsequenceofsymbolrepresentations
(x ,...,x ) to another sequence of equal length (z ,...,z ), with x ,z ∈ Rd, such as a hidden
1 n 1 n i i
layerinatypicalsequencetransductionencoderordecoder. Motivatingouruseofself-attentionwe
considerthreedesiderata.
Oneisthetotalcomputationalcomplexityperlayer. Anotheristheamountofcomputationthatcan
beparallelized,asmeasuredbytheminimumnumberofsequentialoperationsrequired.
Thethirdisthepathlengthbetweenlong-rangedependenciesinthenetwork. Learninglong-range
dependenciesisakeychallengeinmanysequencetransductiontasks. Onekeyfactoraffectingthe
abilitytolearnsuchdependenciesisthelengthofthepathsforwardandbackwardsignalshaveto
traverseinthenetwork. Theshorterthesepathsbetweenanycombinationofpositionsintheinput
andoutputsequences,theeasieritistolearnlong-rangedependencies[12]. Hencewealsocompare
themaximumpathlengthbetweenanytwoinputandoutputpositionsinnetworkscomposedofthe
differentlayertypes.
AsnotedinTable1,aself-attentionlayerconnectsallpositionswithaconstantnumberofsequentially
executed operations, whereas a recurrent layer requires O(n) sequential operations. In terms of
computationalcomplexity,self-attentionlayersarefasterthanrecurrentlayerswhenthesequence
6length n is smaller than the representation dimensionality d, which is most often the case with
sentencerepresentationsusedbystate-of-the-artmodelsinmachinetranslations,suchasword-piece
[38]andbyte-pair[31]representations. Toimprovecomputationalperformancefortasksinvolving
verylongsequences,self-attentioncouldberestrictedtoconsideringonlyaneighborhoodofsizerin
theinputsequencecenteredaroundtherespectiveoutputposition. Thiswouldincreasethemaximum
pathlengthtoO(n/r). Weplantoinvestigatethisapproachfurtherinfuturework.
Asingleconvolutionallayerwithkernelwidthk <ndoesnotconnectallpairsofinputandoutput
positions. DoingsorequiresastackofO(n/k)convolutionallayersinthecaseofcontiguouskernels,
orO(log (n))inthecaseofdilatedconvolutions[18], increasingthelengthofthelongestpaths
k
betweenanytwopositionsinthenetwork. Convolutionallayersaregenerallymoreexpensivethan
recurrent layers, by a factor of k. Separable convolutions [6], however, decrease the complexity
considerably, toO(k·n·d+n·d2). Evenwithk = n, however, thecomplexityofaseparable
convolutionisequaltothecombinationofaself-attentionlayerandapoint-wisefeed-forwardlayer,
theapproachwetakeinourmodel.
Assidebenefit,self-attentioncouldyieldmoreinterpretablemodels.Weinspectattentiondistributions
fromourmodelsandpresentanddiscussexamplesintheappendix. Notonlydoindividualattention
headsclearlylearntoperformdifferenttasks,manyappeartoexhibitbehaviorrelatedtothesyntactic
andsemanticstructureofthesentences.
5 Training
Thissectiondescribesthetrainingregimeforourmodels.
5.1 TrainingDataandBatching
We trained on the standard WMT 2014 English-German dataset consisting of about 4.5 million
sentencepairs. Sentenceswereencodedusingbyte-pairencoding[3],whichhasasharedsource-
targetvocabularyofabout37000tokens. ForEnglish-French,weusedthesignificantlylargerWMT
2014English-Frenchdatasetconsistingof36Msentencesandsplittokensintoa32000word-piece
vocabulary[38].Sentencepairswerebatchedtogetherbyapproximatesequencelength.Eachtraining
batchcontainedasetofsentencepairscontainingapproximately25000sourcetokensand25000
targettokens.
5.2 HardwareandSchedule
Wetrainedourmodelsononemachinewith8NVIDIAP100GPUs. Forourbasemodelsusing
thehyperparametersdescribedthroughoutthepaper,eachtrainingsteptookabout0.4seconds. We
trainedthebasemodelsforatotalof100,000stepsor12hours. Forourbigmodels,(describedonthe
bottomlineoftable3),steptimewas1.0seconds. Thebigmodelsweretrainedfor300,000steps
(3.5days).
5.3 Optimizer
WeusedtheAdamoptimizer[20]withβ =0.9,β =0.98andϵ=10−9. Wevariedthelearning
1 2
rateoverthecourseoftraining,accordingtotheformula:
lrate=d−0.5 ·min(step_num−0.5,step_num·warmup_steps−1.5) (3)
model
Thiscorrespondstoincreasingthelearningratelinearlyforthefirstwarmup_stepstrainingsteps,
anddecreasingitthereafterproportionallytotheinversesquarerootofthestepnumber. Weused
warmup_steps=4000.
5.4 Regularization
Weemploythreetypesofregularizationduringtraining:
7Table2: TheTransformerachievesbetterBLEUscoresthanpreviousstate-of-the-artmodelsonthe
English-to-GermanandEnglish-to-Frenchnewstest2014testsatafractionofthetrainingcost.
BLEU TrainingCost(FLOPs)
Model
EN-DE EN-FR EN-DE EN-FR
ByteNet[18] 23.75
Deep-Att+PosUnk[39] 39.2 1.0·1020
GNMT+RL[38] 24.6 39.92 2.3·1019 1.4·1020
ConvS2S[9] 25.16 40.46 9.6·1018 1.5·1020
MoE[32] 26.03 40.56 2.0·1019 1.2·1020
Deep-Att+PosUnkEnsemble[39] 40.4 8.0·1020
GNMT+RLEnsemble[38] 26.30 41.16 1.8·1020 1.1·1021
ConvS2SEnsemble[9] 26.36 41.29 7.7·1019 1.2·1021
Transformer(basemodel) 27.3 38.1 3.3·1018
Transformer(big) 28.4 41.8 2.3·1019
ResidualDropout Weapplydropout[33]totheoutputofeachsub-layer,beforeitisaddedtothe
sub-layerinputandnormalized. Inaddition,weapplydropouttothesumsoftheembeddingsandthe
positionalencodingsinboththeencoderanddecoderstacks. Forthebasemodel,weusearateof
P =0.1.
drop
LabelSmoothing Duringtraining,weemployedlabelsmoothingofvalueϵ = 0.1[36]. This
ls
hurtsperplexity,asthemodellearnstobemoreunsure,butimprovesaccuracyandBLEUscore.
6 Results
6.1 MachineTranslation
OntheWMT2014English-to-Germantranslationtask,thebigtransformermodel(Transformer(big)
inTable2)outperformsthebestpreviouslyreportedmodels(includingensembles)bymorethan2.0
BLEU,establishinganewstate-of-the-artBLEUscoreof28.4. Theconfigurationofthismodelis
listedinthebottomlineofTable3. Trainingtook3.5dayson8P100GPUs. Evenourbasemodel
surpassesallpreviouslypublishedmodelsandensembles,atafractionofthetrainingcostofanyof
thecompetitivemodels.
OntheWMT2014English-to-Frenchtranslationtask,ourbigmodelachievesaBLEUscoreof41.0,
outperformingallofthepreviouslypublishedsinglemodels,atlessthan1/4thetrainingcostofthe
previousstate-of-the-artmodel. TheTransformer(big)modeltrainedforEnglish-to-Frenchused
dropoutrateP =0.1,insteadof0.3.
drop
Forthebasemodels,weusedasinglemodelobtainedbyaveragingthelast5checkpoints,which
werewrittenat10-minuteintervals. Forthebigmodels,weaveragedthelast20checkpoints. We
usedbeamsearchwithabeamsizeof4andlengthpenaltyα = 0.6[38]. Thesehyperparameters
werechosenafterexperimentationonthedevelopmentset. Wesetthemaximumoutputlengthduring
inferencetoinputlength+50,butterminateearlywhenpossible[38].
Table2summarizesourresultsandcomparesourtranslationqualityandtrainingcoststoothermodel
architecturesfromtheliterature. Weestimatethenumberoffloatingpointoperationsusedtotraina
modelbymultiplyingthetrainingtime,thenumberofGPUsused,andanestimateofthesustained
single-precisionfloating-pointcapacityofeachGPU5.
6.2 ModelVariations
ToevaluatetheimportanceofdifferentcomponentsoftheTransformer,wevariedourbasemodel
indifferentways,measuringthechangeinperformanceonEnglish-to-Germantranslationonthe
5Weusedvaluesof2.8,3.7,6.0and9.5TFLOPSforK80,K40,M40andP100,respectively.
8Table3: VariationsontheTransformerarchitecture. Unlistedvaluesareidenticaltothoseofthebase
model. AllmetricsareontheEnglish-to-Germantranslationdevelopmentset,newstest2013. Listed
perplexitiesareper-wordpiece,accordingtoourbyte-pairencoding,andshouldnotbecomparedto
per-wordperplexities.
train PPL BLEU params
N d d h d d P ϵ
model ff k v drop ls steps (dev) (dev) ×106
base 6 512 2048 8 64 64 0.1 0.1 100K 4.92 25.8 65
1 512 512 5.29 24.9
4 128 128 5.00 25.5
(A)
16 32 32 4.91 25.8
32 16 16 5.01 25.4
16 5.16 25.1 58
(B)
32 5.01 25.4 60
2 6.11 23.7 36
4 5.19 25.3 50
8 4.88 25.5 80
(C) 256 32 32 5.75 24.5 28
1024 128 128 4.66 26.0 168
1024 5.12 25.4 53
4096 4.75 26.2 90
0.0 5.77 24.6
0.2 4.95 25.5
(D)
0.0 4.67 25.3
0.2 5.47 25.7
(E) positionalembeddinginsteadofsinusoids 4.92 25.7
big 6 1024 4096 16 0.3 300K 4.33 26.4 213
developmentset,newstest2013. Weusedbeamsearchasdescribedintheprevioussection,butno
checkpointaveraging. WepresenttheseresultsinTable3.
InTable3rows(A),wevarythenumberofattentionheadsandtheattentionkeyandvaluedimensions,
keeping the amount of computation constant, as described in Section 3.2.2. While single-head
attentionis0.9BLEUworsethanthebestsetting,qualityalsodropsoffwithtoomanyheads.
InTable3rows(B),weobservethatreducingtheattentionkeysized hurtsmodelquality. This
k
suggests that determining compatibility is not easy and that a more sophisticated compatibility
functionthandotproductmaybebeneficial. Wefurtherobserveinrows(C)and(D)that,asexpected,
biggermodelsarebetter,anddropoutisveryhelpfulinavoidingover-fitting.Inrow(E)wereplaceour
sinusoidalpositionalencodingwithlearnedpositionalembeddings[9],andobservenearlyidentical
resultstothebasemodel.
6.3 EnglishConstituencyParsing
ToevaluateiftheTransformercangeneralizetoothertasksweperformedexperimentsonEnglish
constituencyparsing. Thistaskpresentsspecificchallenges: theoutputissubjecttostrongstructural
constraints and is significantly longer than the input. Furthermore, RNN sequence-to-sequence
modelshavenotbeenabletoattainstate-of-the-artresultsinsmall-dataregimes[37].
Wetraineda4-layertransformerwithd =1024ontheWallStreetJournal(WSJ)portionofthe
model
PennTreebank[25],about40Ktrainingsentences. Wealsotraineditinasemi-supervisedsetting,
usingthelargerhigh-confidenceandBerkleyParsercorporafromwithapproximately17Msentences
[37]. Weusedavocabularyof16KtokensfortheWSJonlysettingandavocabularyof32Ktokens
forthesemi-supervisedsetting.
Weperformedonlyasmallnumberofexperimentstoselectthedropout,bothattentionandresidual
(section5.4),learningratesandbeamsizeontheSection22developmentset,allotherparameters
remained unchanged from the English-to-German base translation model. During inference, we
9Table4: TheTransformergeneralizeswelltoEnglishconstituencyparsing(ResultsareonSection23
ofWSJ)
Parser Training WSJ23F1
Vinyals&Kaiserelal. (2014)[37] WSJonly,discriminative 88.3
Petrovetal. (2006)[29] WSJonly,discriminative 90.4
Zhuetal. (2013)[40] WSJonly,discriminative 90.4
Dyeretal. (2016)[8] WSJonly,discriminative 91.7
Transformer(4layers) WSJonly,discriminative 91.3
Zhuetal. (2013)[40] semi-supervised 91.3
Huang&Harper(2009)[14] semi-supervised 91.3
McCloskyetal. (2006)[26] semi-supervised 92.1
Vinyals&Kaiserelal. (2014)[37] semi-supervised 92.1
Transformer(4layers) semi-supervised 92.7
Luongetal. (2015)[23] multi-task 93.0
Dyeretal. (2016)[8] generative 93.3
increasedthemaximumoutputlengthtoinputlength+300. Weusedabeamsizeof21andα=0.3
forbothWSJonlyandthesemi-supervisedsetting.
Our results in Table 4 show that despite the lack of task-specific tuning our model performs sur-
prisinglywell,yieldingbetterresultsthanallpreviouslyreportedmodelswiththeexceptionofthe
RecurrentNeuralNetworkGrammar[8].
IncontrasttoRNNsequence-to-sequencemodels[37],theTransformeroutperformstheBerkeley-
Parser[29]evenwhentrainingonlyontheWSJtrainingsetof40Ksentences.
7 Conclusion
Inthiswork,wepresentedtheTransformer,thefirstsequencetransductionmodelbasedentirelyon
attention,replacingtherecurrentlayersmostcommonlyusedinencoder-decoderarchitectureswith
multi-headedself-attention.
For translation tasks, the Transformer can be trained significantly faster than architectures based
on recurrent or convolutional layers. On both WMT 2014 English-to-German and WMT 2014
English-to-Frenchtranslationtasks,weachieveanewstateoftheart. Intheformertaskourbest
modeloutperformsevenallpreviouslyreportedensembles.
Weareexcitedaboutthefutureofattention-basedmodelsandplantoapplythemtoothertasks. We
plantoextendtheTransformertoproblemsinvolvinginputandoutputmodalitiesotherthantextand
toinvestigatelocal,restrictedattentionmechanismstoefficientlyhandlelargeinputsandoutputs
suchasimages,audioandvideo. Makinggenerationlesssequentialisanotherresearchgoalsofours.
The code we used to train and evaluate our models is available at https://github.com/
tensorflow/tensor2tensor.
Acknowledgements WearegratefultoNalKalchbrennerandStephanGouwsfortheirfruitful
comments,correctionsandinspiration.
'''

In [2]:
from heirarchical_retrieval import Parent_retrieval

c:\Users\samarth.srivastava\Desktop\new_check_proj\test_env2\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
p = Parent_retrieval(collection_name='some_collection', username='test_user', auto_id=True)

collection inserted some_collection


In [4]:
ret = p.get_retriever(raw_text=raw_text)

[{'doc_id': '62d366ee-7329-4dc5-96f6-c4615fcd1feb'}, {'doc_id': '62d366ee-7329-4dc5-96f6-c4615fcd1feb'}, {'doc_id': '62d366ee-7329-4dc5-96f6-c4615fcd1feb'}, {'doc_id': '62d366ee-7329-4dc5-96f6-c4615fcd1feb'}, {'doc_id': '62d366ee-7329-4dc5-96f6-c4615fcd1feb'}, {'doc_id': '62d366ee-7329-4dc5-96f6-c4615fcd1feb'}, {'doc_id': '62d366ee-7329-4dc5-96f6-c4615fcd1feb'}, {'doc_id': '62d366ee-7329-4dc5-96f6-c4615fcd1feb'}, {'doc_id': '62d366ee-7329-4dc5-96f6-c4615fcd1feb'}, {'doc_id': '62d366ee-7329-4dc5-96f6-c4615fcd1feb'}, {'doc_id': '62d366ee-7329-4dc5-96f6-c4615fcd1feb'}, {'doc_id': '62d366ee-7329-4dc5-96f6-c4615fcd1feb'}, {'doc_id': '62d366ee-7329-4dc5-96f6-c4615fcd1feb'}, {'doc_id': '62d366ee-7329-4dc5-96f6-c4615fcd1feb'}, {'doc_id': '62d366ee-7329-4dc5-96f6-c4615fcd1feb'}, {'doc_id': '62d366ee-7329-4dc5-96f6-c4615fcd1feb'}, {'doc_id': '62d366ee-7329-4dc5-96f6-c4615fcd1feb'}, {'doc_id': '62d366ee-7329-4dc5-96f6-c4615fcd1feb'}, {'doc_id': '62d366ee-7329-4dc5-96f6-c4615fcd1feb'}, {'doc_id': 

2025-11-07 13:52:58,217 [ERROR][handler]: RPC error: [insert_rows], <DataNotMatchException: (code=1, message=Insert missed an field `pk` to collection without set nullable==true or set default_value)>, <Time:{'RPC start': '2025-11-07 13:52:58.217299', 'RPC error': '2025-11-07 13:52:58.217299'}> (decorators.py:140)
Failed to insert batch starting at entity: 0/1998


DataNotMatchException: <DataNotMatchException: (code=1, message=Insert missed an field `pk` to collection without set nullable==true or set default_value)>